SWIN Standalone Model

In [1]:
!pip install colorama

In [2]:
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d viswachaitanyasai/deepfake-dataset
!unzip deepfake-dataset.zip -d /content/dataset

Streaming output truncated to the last 5000 lines.
  inflating: /content/dataset/deepfake_dataset/validation/ffpp/fake/validation_3505.jpg  
  inflating: /content/dataset/deepfake_dataset/validation/ffpp/fake/validation_3506.jpg  
  inflating: /content/dataset/deepfake_dataset/validation/ffpp/fake/validation_3507.jpg  
  inflating: /content/dataset/deepfake_dataset/validation/ffpp/fake/validation_3508.jpg  
  inflating: /content/dataset/deepfake_dataset/validation/ffpp/fake/validation_3509.jpg  
  inflating: /content/dataset/deepfake_dataset/validation/ffpp/fake/validation_3510.jpg  
  inflating: /content/dataset/deepfake_dataset/validation/ffpp/fake/validation_3511.jpg  
  inflating: /content/dataset/deepfake_dataset/validation/ffpp/fake/validation_3512.jpg  
  inflating: /content/dataset/deepfake_dataset/validation/ffpp/fake/validation_3513.jpg  
  inflating: /content/dataset/deepfake_dataset/validation/ffpp/fake/validation_3514.jpg  
  inflating: /content/dataset/deepfake_dataset/va

In [ ]:
#!/usr/bin/env python
import os
import sys
import time
import copy
import random
import pickle
import logging
import argparse
import numpy as np
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets, transforms
import timm
from tqdm.auto import tqdm
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score,
                             recall_score, f1_score, roc_auc_score, classification_report)
from colorama import Fore, Style
import matplotlib.pyplot as plt

##############################################
#            Setup & Logging                 #
##############################################
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logging.info(f"Using device: {device}")
if torch.cuda.is_available():
    logging.info(f"GPU: {torch.cuda.get_device_name(0)} - Allocated Memory: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

green_bar_format = f"{Fore.GREEN}{{l_bar}}{{bar}}{{r_bar}}{Style.RESET_ALL}"

##############################################
#         Advanced Data Augmentation         #
##############################################
from albumentations import (
    HorizontalFlip, VerticalFlip, ShiftScaleRotate, CLAHE,
    RandomRotate90, Transpose, HueSaturationValue, GaussNoise,
    Sharpen, Emboss, RandomBrightnessContrast, OneOf, Compose, RandomCrop
)
from albumentations.pytorch import ToTensorV2
from PIL import Image

def strong_aug(p=0.5):
    return Compose([
        RandomRotate90(p=0.2),
        Transpose(p=0.2),
        HorizontalFlip(p=0.5),
        VerticalFlip(p=0.5),
        OneOf([GaussNoise(),], p=0.2),
        ShiftScaleRotate(p=0.2),
        OneOf([
            CLAHE(clip_limit=2),
            Sharpen(),
            Emboss(),
            RandomBrightnessContrast(),
        ], p=0.2),
        HueSaturationValue(p=0.2),
        RandomCrop(width=200, height=200, p=0.3),
    ], p=p)

def augment(aug, image):
    return aug(image=image)["image"]

class Aug(object):
    def __call__(self, img):
        aug = strong_aug(p=0.9)
        img_np = np.array(img)
        augmented = augment(aug, img_np)
        return Image.fromarray(augmented)

# Label smoothing BCE loss
class SmoothBCEWithLogitsLoss(nn.Module):
    def __init__(self, smoothing=0.1, pos_weight=None):
        super(SmoothBCEWithLogitsLoss, self).__init__()
        self.smoothing = smoothing
        self.pos_weight = pos_weight

    def forward(self, logits, target):
        target = target * (1 - self.smoothing) + 0.5 * self.smoothing
        return nn.functional.binary_cross_entropy_with_logits(logits, target, pos_weight=self.pos_weight)

def normalize_data():
    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]
    return {
        "train": transforms.Compose([
            Aug(),
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
            transforms.RandomErasing(p=0.2)
        ]),
        "valid": transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std)
        ]),
        "test": transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std)
        ])
    }

##############################################
#            Dataset Loading               #
##############################################
base_dataset_dir = "/content/dataset/deepfake_dataset"
train_datasets = []
val_datasets = []
test_datasets = []
sub_datasets = ["celebdf", "dfdc", "ffpp", "realfake"]

data_transforms = normalize_data()

for sub in sub_datasets:
    train_path = f"{base_dataset_dir}/train/{sub}"
    train_dataset = datasets.ImageFolder(train_path, transform=data_transforms["train"])
    train_datasets.append(train_dataset)

    val_path = f"{base_dataset_dir}/validation/{sub}"
    val_dataset = datasets.ImageFolder(val_path, transform=data_transforms["valid"])
    val_datasets.append(val_dataset)

    test_path = f"{base_dataset_dir}/test/{sub}"
    test_dataset = datasets.ImageFolder(test_path, transform=data_transforms["test"])
    test_datasets.append(test_dataset)

full_train_dataset = ConcatDataset(train_datasets)
full_val_dataset = ConcatDataset(val_datasets)
test_dataset = ConcatDataset(test_datasets)

train_subset_size = 20000
train_indices = random.sample(range(len(full_train_dataset)), train_subset_size)
train_dataset = Subset(full_train_dataset, train_indices)
print(f"Using {len(train_dataset)} training samples.")

val_subset_size = 5000
val_indices = random.sample(range(len(full_val_dataset)), val_subset_size)
val_dataset = Subset(full_val_dataset, val_indices)
print(f"Using {len(val_dataset)} validation samples.")

def count_labels_from_subset(subset):
    all_targets = []
    for idx in subset.indices:
        for dataset in subset.dataset.datasets:
            if idx < len(dataset):
                label = dataset.targets[idx]
                all_targets.append(label)
                break
            else:
                idx -= len(dataset)
    return Counter(all_targets)

def count_labels_from_concat(concat_dataset):
    all_targets = []
    for dataset in concat_dataset.datasets:
        all_targets.extend(dataset.targets)
    return Counter(all_targets)

train_label_counts = count_labels_from_subset(train_dataset)
print(f"Train Dataset Class Counts: {train_label_counts}")
val_label_counts = count_labels_from_subset(val_dataset)
print(f"Validation Dataset Class Counts: {val_label_counts}")
test_label_counts = count_labels_from_concat(test_dataset)
print(f"Test Dataset Class Counts: {test_label_counts}")

batch_size = 16
num_workers = 2

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=True, persistent_workers=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                         num_workers=num_workers, pin_memory=True)

dataset_sizes = {"train": len(train_dataset), "validation": len(val_dataset), "test": len(test_dataset)}
print(f"Dataset sizes -> Train: {dataset_sizes['train']}, Validation: {dataset_sizes['validation']}, Test: {dataset_sizes['test']}")

##############################################
#         Swin Transformer Model Definition  #
##############################################
class SwinDeepfakeDetector(nn.Module):
    def __init__(self, num_classes=1):
        super(SwinDeepfakeDetector, self).__init__()
        # Swin Transformer Small model
        self.swin = timm.create_model('swin_small_patch4_window7_224', pretrained=True)

        for param in self.swin.parameters():
            param.requires_grad = True

        # Removing classification head
        if hasattr(self.swin, 'head'):
            self.swin.head = nn.Identity()
        elif hasattr(self.swin, 'classifier'):
            self.swin.classifier = nn.Identity()

        dummy = torch.randn(1, 3, 224, 224)
        with torch.no_grad():
            features = self.swin.forward_features(dummy) if hasattr(self.swin, 'forward_features') else self.swin(dummy)
            if features.ndim == 4:
                features = nn.functional.adaptive_avg_pool2d(features, 1).view(1, -1)
            self.feature_dim = features.shape[1]
            logging.info(f"Swin Transformer feature dim: {self.feature_dim}")
            print(f"Swin Transformer feature dim: {self.feature_dim}")

        self.classifier = nn.Sequential(
            nn.Linear(self.feature_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        features = self.swin.forward_features(x) if hasattr(self.swin, 'forward_features') else self.swin(x)
        if features.ndim == 4:
            features = nn.functional.adaptive_avg_pool2d(features, 1).view(x.size(0), -1)
        out = self.classifier(features)
        return out

model = SwinDeepfakeDetector(num_classes=1)
print("Swin Transformer Deepfake Detector created.")

##############################################
#        Optimizer, Scheduler Setup          #
##############################################
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)
num_epochs = 15 

def get_lr_scheduler(optimizer, num_epochs, warmup_epochs=2):
    def lr_lambda(current_epoch):
        if current_epoch < warmup_epochs:
            return float(current_epoch + 1) / warmup_epochs
        else:
            return 0.5 * (1 + np.cos(np.pi * (current_epoch - warmup_epochs) / (num_epochs - warmup_epochs)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

scheduler = get_lr_scheduler(optimizer, num_epochs=num_epochs, warmup_epochs=2)

pos_weight_value = 10667 / 9333
criterion = SmoothBCEWithLogitsLoss(smoothing=0.1, pos_weight=torch.tensor([pos_weight_value]).to(device))

##############################################
#          Mixup Data Augmentation           #
##############################################
def mixup_data(x, y, alpha=0.4):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

##############################################
#         Training & Validation Loops        #
##############################################
model = model.to(device)
scaler = torch.amp.GradScaler('cuda')
best_val_loss = float('inf')
patience = 5
trigger_times = 0

train_history = {"loss": [], "acc": []}
val_history = {"loss": [], "acc": []}

def optimize_threshold(probs, true_labels):
    best_thresh = 0.5
    best_acc = 0.0
    for t in np.arange(0.0, 1.0, 0.01):
        preds = (probs >= t).astype(int)
        acc = accuracy_score(true_labels, preds)
        if acc > best_acc:
            best_acc = acc
            best_thresh = t
    return best_thresh, best_acc

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    train_correct = 0
    train_total = 0
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Training", leave=False, bar_format=green_bar_format)
    for images, labels in train_bar:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)
        optimizer.zero_grad()

        if random.random() < 0.5:
            images, targets_a, targets_b, lam = mixup_data(images, labels)
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
        else:
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(outputs) > 0.5).int()
        train_correct += (preds.cpu() == labels.cpu().int()).sum().item()
        train_total += labels.size(0)
        train_bar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{(train_correct/train_total):.4f}")
    epoch_loss = running_loss / dataset_sizes["train"]
    train_acc = train_correct / train_total
    train_history["loss"].append(epoch_loss)
    train_history["acc"].append(train_acc)

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    val_probs = []
    val_true = []
    val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Validation", leave=False, bar_format=green_bar_format)
    with torch.no_grad():
        for images, labels in val_bar:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            probs = torch.sigmoid(outputs)
            val_probs.extend(probs.cpu().numpy().flatten())
            preds = (probs > 0.5).int()
            val_correct += (preds.cpu() == labels.cpu().int()).sum().item()
            val_total += labels.size(0)
            val_true.extend(labels.cpu().numpy().flatten())
            val_bar.set_postfix(loss=f"{loss.item():.4f}")
    epoch_val_loss = val_loss / dataset_sizes["validation"]
    epoch_val_acc = val_correct / val_total
    val_history["loss"].append(epoch_val_loss)
    val_history["acc"].append(epoch_val_acc)
    scheduler.step()

    best_thresh, best_val_acc = optimize_threshold(np.array(val_probs), np.array(val_true))
    val_preds = (np.array(val_probs) >= best_thresh).astype(int)
    val_accuracy = accuracy_score(val_true, val_preds)
    val_precision = precision_score(val_true, val_preds)
    val_recall = recall_score(val_true, val_preds)
    val_f1 = f1_score(val_true, val_preds)
    try:
        val_auc = roc_auc_score(val_true, val_probs)
    except Exception:
        val_auc = 0.0

    logging.info(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {epoch_loss:.4f}, Train Acc: {train_acc:.4f}")
    logging.info(f"                Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f}, Best Thresh: {best_thresh:.2f}")
    logging.info(f"                Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}, AUC: {val_auc:.4f}")
    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {epoch_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"                Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f}, Best Thresh: {best_thresh:.2f}")
    print(f"                Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}, AUC: {val_auc:.4f}")

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        trigger_times = 0
        torch.save(model.state_dict(), "best_swin_model.pth")
    else:
        trigger_times += 1
        if trigger_times >= patience:
            print("Early stopping triggered!")
            break

def save_plot(metric_history, metric_name, mod):
    plt.figure(figsize=(10,5))
    plt.plot(metric_history, label=f"{metric_name}")
    plt.title(f"{metric_name} Trend")
    plt.xlabel("Epoch")
    plt.ylabel(metric_name)
    plt.legend()
    os.makedirs("result/figs", exist_ok=True)
    filename = os.path.join("result/figs", f"{mod}_{metric_name}_trend_{time.strftime('%b_%d_%Y_%H_%M_%S')}.png")
    plt.savefig(filename)
    plt.close()
    print(f"{metric_name} plot saved as {filename}")

save_plot(train_history["loss"], "Train Loss", "swin")
save_plot(train_history["acc"], "Train Accuracy", "swin")
save_plot(val_history["loss"], "Validation Loss", "swin")
save_plot(val_history["acc"], "Validation Accuracy", "swin")

##############################################
#          Testing & Evaluation              #
##############################################
model.load_state_dict(torch.load("best_swin_model.pth"))
model.eval()

test_probs = []
all_preds = []
all_labels = []
test_loss = 0.0
test_correct = 0
test_total = 0
test_bar = tqdm(test_loader, desc="Testing", leave=False, bar_format=green_bar_format)
with torch.no_grad():
    for images, labels in test_bar:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)
        test_loss += loss.item() * images.size(0)
        probs = torch.sigmoid(outputs)
        test_probs.extend(probs.cpu().numpy().flatten())
        preds = (probs > best_thresh).int()
        all_preds.extend(preds.cpu().numpy().flatten())
        all_labels.extend(labels.cpu().numpy().flatten())
        test_correct += (preds.cpu() == labels.cpu().int()).sum().item()
        test_total += labels.size(0)
        test_bar.set_postfix(loss=f"{loss.item():.4f}")
test_loss /= dataset_sizes["test"]
test_acc = test_correct / test_total

final_preds = (np.array(test_probs) >= best_thresh).astype(int)
final_labels = np.array(all_labels).astype(int)
accuracy_metric = accuracy_score(final_labels, final_preds)
precision_metric = precision_score(final_labels, final_preds)
recall_metric = recall_score(final_labels, final_preds)
f1_metric = f1_score(final_labels, final_preds)
conf_matrix = confusion_matrix(final_labels, final_preds)
try:
    auc_metric = roc_auc_score(final_labels, test_probs)
except Exception:
    auc_metric = "N/A"
classification_rep = classification_report(final_labels, final_preds)

print("\n=== Test Evaluation Metrics ===")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {accuracy_metric:.4f}")
print(f"Precision: {precision_metric:.4f}")
print(f"Recall: {recall_metric:.4f}")
print(f"F1 Score: {f1_metric:.4f}")
print(f"ROC-AUC: {auc_metric}")
print("Confusion Matrix:")
print(conf_matrix)
print("Classification Report:")
print(classification_rep)

Using 20000 training samples.
Using 5000 validation samples.
Train Dataset Class Counts: Counter({0: 10633, 1: 9367})
Validation Dataset Class Counts: Counter({0: 2675, 1: 2325})
Test Dataset Class Counts: Counter({0: 4470, 1: 3874})
Dataset sizes -> Train: 20000, Validation: 5000, Test: 8344


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/200M [00:00<?, ?B/s]

Swin Transformer feature dim: 7
Swin Transformer Deepfake Detector created.


Epoch 1/15 - Training:   0%|          | 0/1250 [00:00<?, ?it/s]

...

Epoch 15/15 - Training:   0%|          | 0/1250 [00:00<?, ?it/s]

Epoch 15/15 - Validation:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch [15/15] - Train Loss: 0.5088, Train Acc: 0.7715
                Val Loss: 0.3710, Val Acc: 0.9020, Best Thresh: 0.44
                Precision: 0.8959, Recall: 0.8994, F1: 0.8976, AUC: 0.9707
Train Loss plot saved as result/figs/swin_Train Loss_trend_Apr_01_2025_17_00_40.png
Train Accuracy plot saved as result/figs/swin_Train Accuracy_trend_Apr_01_2025_17_00_40.png
Validation Loss plot saved as result/figs/swin_Validation Loss_trend_Apr_01_2025_17_00_40.png
Validation Accuracy plot saved as result/figs/swin_Validation Accuracy_trend_Apr_01_2025_17_00_40.png


Testing:   0%|          | 0/522 [00:00<?, ?it/s]


=== Test Evaluation Metrics ===
Test Loss: 0.3803
Test Accuracy: 0.8991
Precision: 0.8828
Recall: 0.9024
F1 Score: 0.8925
ROC-AUC: 0.9667480328328939
Confusion Matrix:
[[4006  464]
 [ 378 3496]]
Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.90      0.90      4470
           1       0.88      0.90      0.89      3874

    accuracy                           0.90      8344
   macro avg       0.90      0.90      0.90      8344
weighted avg       0.90      0.90      0.90      8344

